# Colab runner

Runs the project on a Colab GPU from VS Code.

**Every new session:** run cells 1 and 2 first, in order.

- Code comes from GitHub (always the latest pushed version).
- Big files (`data/`, `outputs/`, `judges/`) live on Google Drive, so they survive when the session ends.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the latest code and install it

Paste your GitHub repository address between the quotes on the first line (once).

In [ ]:
REPO = "https://github.com/tca-codes/moral-rag-cf.git"   # e.g. "https://github.com/your-username/moral-rag-cf.git"
DRIVE_DIR = "/content/drive/MyDrive/moral-rag-cf"

import os
assert REPO, "Paste your GitHub repository address into REPO on the first line of this cell."
assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not connected. Run cell 1 first."

if not os.path.isdir("/content/moral-rag-cf/.git"):
    !git clone -q {REPO} /content/moral-rag-cf
%cd /content/moral-rag-cf
!git pull -q
!pip install -q -r requirements.txt
!pip install -q -e .

# data, outputs and judges are stored on Drive and linked into the code folder
for name in ["data", "outputs", "judges"]:
    os.makedirs(f"{DRIVE_DIR}/{name}", exist_ok=True)
    if not os.path.exists(name):
        os.symlink(f"{DRIVE_DIR}/{name}", name)

print("\nCode version in use:")
!git log -1 --format="%h  %s  (%cr)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Speed test (40 questions, 7B model, uncompressed)

Use this on an L4 or A100. On a T4, remove `--no-4bit` and change 16 to 2.

In [ ]:
!python scripts/run_agents.py --split test --n 40 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 32 --out outputs/speedtest.jsonl

## When you are done

Disconnect the runtime so it stops using compute units: open the Command Palette and run **Colab: Remove Server**.